# Data Cleaning Script — EXMD 601 Project
## Research Question: What factors predict cardiovascular complications in diabetic patients?

This notebook cleans the synthetic MUHC diabetic patient dataset (from MDClone) and prepares it for modeling.

**Key decisions:**
- Filter to adults only (age >= 18)
- Drop homocysteine (nearly 100% missing)
- For lab values with high missingness, create binary "has_lab_test" indicator columns so the model can learn from missingness patterns, then impute remaining NaN values with median
- Drop 457 rows with "censored" gender (small fraction of 50k+ patients)
- Include additional clinically relevant features: age, encounter type, MI history, death info
- Two outcome variables:
  - **Outcome A**: Stroke OR Heart Failure OR CKD within 10 years
  - **Outcome B**: ED visit within 1 year

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

### 2. Load data

In [ ]:
# Use relative path from this notebook's location
data_path = '../../data/data1.csv'
data = pd.read_csv(data_path)

print(f"Raw dataset: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"\nGender distribution:\n{data['gender'].value_counts(dropna=False)}")
print(f"\nAge range: {data['r_age_at_event'].min()} - {data['r_age_at_event'].max()}")

### 3. Filter: adults only and remove censored gender

In [ ]:
# Filter to age >= 18
data = data[data['r_age_at_event'] >= 18].copy()
print(f"After age >= 18 filter: {data.shape[0]} rows")

# Remove rows where gender is "censored" (only ~457 rows out of 50k+)
data = data[data['gender'] != 'censored'].copy()
print(f"After removing censored gender: {data.shape[0]} rows")

### 4. Define outcome variables

- **Outcome A**: Composite cardiovascular outcome — stroke OR heart failure OR CKD within 10 years (3652.5 days). NaN in the condition date column means the event never occurred, so we treat it as False.
- **Outcome B**: ED visit within 1 year (365.25 days). NaN = no ED visit = False.

In [ ]:
days_10_years = 3652.5
days_1_year = 365.25

# Outcome A: stroke OR heart failure OR CKD within 10 years
# NaN means the condition never occurred -> fillna with a large number so comparison is False
stroke_col = 'c_20yPost_avc_like_date_time_of_condition_assigned_days_from_reference'
hf_col = 'c_20yPost_congestive_heart_failure_date_time_of_condition_assigned_days_from_reference'
ckd_col = 'c_20yPost_chonic_kidney_disease_date_time_of_condition_assigned_days_from_reference'

data['stroke_10y'] = data[stroke_col].fillna(999999) <= days_10_years
data['hf_10y'] = data[hf_col].fillna(999999) <= days_10_years
data['ckd_10y'] = data[ckd_col].fillna(999999) <= days_10_years

data['outcome_a'] = (data['stroke_10y'] | data['hf_10y'] | data['ckd_10y']).astype(int)

# Outcome B: ED visit within 1 year
ed_col = 'e_30dPost_admission_start_date_days_from_reference'
data['outcome_b'] = (data[ed_col].fillna(999999) <= days_1_year).astype(int)

print("Outcome A (stroke/HF/CKD within 10y):")
print(data['outcome_a'].value_counts())
print(f"\nOutcome B (ED visit within 1y):")
print(data['outcome_b'].value_counts())

# Clean up temporary columns
data.drop(columns=['stroke_10y', 'hf_10y', 'ckd_10y'], inplace=True)

### 5. Feature engineering

**Features selected and justification:**
- `r_age_at_event` — Age is a strong known predictor of cardiovascular risk
- `gender` — Sex differences in cardiovascular disease are well-established
- Lab values (HbA1c, Cholesterol, Glucose, Triglycerides, Creatinine) — Standard clinical markers for diabetes management and cardiovascular risk
- `has_*` indicators — Whether the lab test was ordered captures clinical decision-making and disease severity
- `has_mi_20y` — Prior MI is a major cardiovascular risk factor
- `encounter_inpatient` — Inpatient encounters suggest higher disease acuity
- `has_death_record` — Captures overall mortality risk in dataset
- `hospitalized` — Whether the patient had a hospitalization after reference date

**Dropped:** Homocysteine (nearly 100% missing — no usable signal)

In [ ]:
# --- Gender: encode as binary ---
data['gender'] = data['gender'].map({'Male': 0, 'Female': 1})

# --- Age ---
# r_age_at_event is already numeric, use directly

# --- Lab values: create missingness indicators + median imputation ---
# We DROP homocysteine (nearly 100% missing)
lab_cols = {
    'l_90dPost_hba1c_result_numeric_calculated': 'hba1c',
    'l_90dPost_cholesterol_total_result_numeric_calculated': 'cholesterol',
    'l_90dPost_glucose_result_numeric_calculated': 'glucose',
    'l_90dPost_triglycerides_result_numeric_calculated': 'triglycerides',
    'l_90dPost_creatinine_result_numeric_calculated': 'creatinine',
}

# Report missingness before imputation
print("Lab value missingness:")
for orig_col, short_name in lab_cols.items():
    pct_missing = data[orig_col].isna().mean() * 100
    print(f"  {short_name}: {pct_missing:.1f}% missing")
    
    # Create binary indicator: 1 = lab test was available, 0 = missing
    data[f'has_{short_name}'] = data[orig_col].notna().astype(int)

# Median imputation for the lab values
lab_orig_cols = list(lab_cols.keys())
imputer = SimpleImputer(strategy='median')
data[lab_orig_cols] = imputer.fit_transform(data[lab_orig_cols])

# --- MI within 20 years: binary flag ---
mi_col = 'c_20yPost_myocardial_infarction_date_time_of_condition_assigned_days_from_reference'
data['has_mi_20y'] = data[mi_col].notna().astype(int)
print(f"\nMI within 20y: {data['has_mi_20y'].sum()} patients ({data['has_mi_20y'].mean()*100:.1f}%)")

# --- Encounter type: binary for inpatient ---
data['encounter_inpatient'] = (data['r_encounter_types'] == 'inpatient').astype(int)
print(f"Inpatient encounters: {data['encounter_inpatient'].sum()} ({data['encounter_inpatient'].mean()*100:.1f}%)")

# --- Death record: binary ---
data['has_death_record'] = data['d_death_date_days_from_reference'].notna().astype(int)
print(f"Has death record: {data['has_death_record'].sum()} ({data['has_death_record'].mean()*100:.1f}%)")

# --- Hospitalization: binary ---
data['hospitalized'] = data['h_afterRef_facility_arrival_date_time_days_from_reference'].notna().astype(int)
print(f"Hospitalized: {data['hospitalized'].sum()} ({data['hospitalized'].mean()*100:.1f}%)")

### 6. Assemble final cleaned dataset

In [ ]:
# Select and rename feature columns for the final dataset
feature_columns = {
    'r_age_at_event': 'age',
    'gender': 'gender',
    # Lab values (imputed)
    'l_90dPost_hba1c_result_numeric_calculated': 'hba1c',
    'l_90dPost_cholesterol_total_result_numeric_calculated': 'cholesterol',
    'l_90dPost_glucose_result_numeric_calculated': 'glucose',
    'l_90dPost_triglycerides_result_numeric_calculated': 'triglycerides',
    'l_90dPost_creatinine_result_numeric_calculated': 'creatinine',
    # Missingness indicators
    'has_hba1c': 'has_hba1c',
    'has_cholesterol': 'has_cholesterol',
    'has_glucose': 'has_glucose',
    'has_triglycerides': 'has_triglycerides',
    'has_creatinine': 'has_creatinine',
    # Clinical features
    'has_mi_20y': 'has_mi_20y',
    'encounter_inpatient': 'encounter_inpatient',
    'has_death_record': 'has_death_record',
    'hospitalized': 'hospitalized',
}

# Build final dataframe
cleaned_df = data[list(feature_columns.keys())].rename(columns=feature_columns).copy()
cleaned_df['outcome_a'] = data['outcome_a'].values
cleaned_df['outcome_b'] = data['outcome_b'].values

print(f"Final cleaned dataset: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")
print(f"\nColumns: {list(cleaned_df.columns)}")
print(f"\nMissing values per column:\n{cleaned_df.isnull().sum()}")
print(f"\nFirst 5 rows:")
cleaned_df.head()

### 7. Save cleaned dataset

In [ ]:
output_path = '../../data/cleaned_data.csv'
cleaned_df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to {output_path}")
print(f"Shape: {cleaned_df.shape}")
print(f"\nDescriptive statistics:")
cleaned_df.describe()